In [1]:
import os
from openai import OpenAI
import pandas as pd
import numpy as np
import re
from pypinyin import lazy_pinyin
from rapidfuzz import fuzz
from mutagen.mp3 import MP3
import math
from uuid import uuid4 as uuid
from scipy.optimize import linear_sum_assignment
from dotenv import load_dotenv
from pydub import AudioSegment
load_dotenv(".env")

root = "/mnt/NextcloudSacmData/sacm.av/files/Recordings"

In [14]:
def get_embedding(text):
    text = re.sub(r"[，。！？、“”：；\n]", " ", text)
    response = client.embeddings.create(
        model="text-embedding-3-large",
        input=text
    )
    return response.data[0].embedding


def pinyin(text):
    text = re.sub(r"[，。！？、“”：；\n]", " ", text)
    result = " ".join(lazy_pinyin(text))
    return re.sub(r"\s+", " ", result).strip()


def log_score(x, k=0.1):
    return math.log(1 + k*x) / math.log(1 + 100*k)


def windows(tokens, size, step):
    if len(tokens) <= size:
        yield " ".join(tokens)
    else:
        for i in range(0, len(tokens) - size + 1, step):
            yield " ".join(tokens[i:i+size])

def best_window_score(query_py, lyrics_py, size=50, step=10):
    query_tokens = query_py.split()
    lyric_tokens = lyrics_py.split()
    score = max(
        fuzz.ratio(qw, lw)
        for qw in windows(query_tokens, size, step)
        for lw in windows(lyric_tokens, size, step)
    )
    return score / 100


def weighted_avg(a, b, alpha=0.5, beta=0.5):
    return (a * alpha + b * beta) / 2


def match_zoom_to_sq(d):
    files = sorted(os.listdir(f"{root}/{d}"))
    zoom_files = {f: os.path.getsize(f"{root}/{d}/{f}") for f in files if f.startswith("ZOOM")}
    other_files = {f: os.path.getsize(f"{root}/{d}/{f}") for f in files if not f.startswith("ZOOM")}
    if not zoom_files:
        return {}
    zoom_items = list(zoom_files.items())
    other_items = list(other_files.items())
    cost = np.array([
        [abs(z_size - o_size) for _, o_size in other_items]
        for _, z_size in zoom_items
    ])
    rows, cols = linear_sum_assignment(cost)
    matches = {
        zoom_items[r][0]: other_items[c][0]
        for r, c in zip(rows, cols)
    }
    return matches


def crop_to_limit(filepath, limit=26_214_400, margin=0.90):
    """Return a path to an audio file <= `limit` bytes for Whisper's 25 MiB cap.

    If the file is already under the limit it's returned unchanged. Otherwise a
    centered segment (head/tail dropped evenly) is exported near the original
    bitrate so the result lands just under the cap.
    """
    size = os.path.getsize(filepath)
    if size <= limit:
        return None

    audio = AudioSegment.from_file(filepath)
    dur_ms = len(audio)
    keep_ms = int(dur_ms * (limit / size) * margin)   # fraction of runtime that fits
    start = max(0, (dur_ms - keep_ms) // 2)            # center the crop
    cropped = audio[start:start + keep_ms]

    bitrate_kbps = int(size * 8 / (dur_ms / 1000) / 1000)
    crop_path = f"tmp/{uuid()}.mp3"
    cropped.export(crop_path, format="mp3", bitrate=f"{bitrate_kbps}k")
    print(f"cropped {size:,} → {os.path.getsize(crop_path):,} bytes (limit {limit:,}) → {crop_path}")
    return crop_path

In [8]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
df = pd.read_pickle("song_embeddings_large.pkl")
embeddings = np.vstack(df["embedding"].values).astype(np.float32)
embeddings /= np.linalg.norm(embeddings, axis=1, keepdims=True)
df.head()

,code,type,title,lyrics,pinyin,embedding
0,R7-1,PnW,若有人在基督里,我来到主座前，生命交出你掌权。\n你心意向我彰显，我愿成为你荣耀的器皿。\n旧事已过去是新的...,wo lai dao zhu zuo qian sheng ming jiao chu ni...,"[0.0189666748046875, 0.005886077880859375, -0...."
1,A2-1,PnW,安静,藏我在翅膀荫下\n遮盖我在你大能手中\n当大海翻腾波涛汹涌\n我随你展翅暴风上空\n父你仍作...,cang wo zai chi bang yin xia zhe gai wo zai ni...,"[0.029296875, -0.0100555419921875, 0.001227378..."
2,A4-1,PnW,爱的真谛,爱是恒久忍耐，又有恩慈，爱是不嫉妒。\n爱是不自夸，不张狂，不做害羞的事。\n不求自己的益处...,ai shi heng jiu ren nai you you en ci ai shi b...,"[-0.042236328125, -0.04388427734375, -0.007007..."
3,A4-2,PnW,爱中相遇,每一天，渴望与祢在爱中相遇。再一次，将自己完全地献给祢。唯有祢，是我的喜乐和力量，我甘愿舍弃...,mei yi tian ke wang yu mi zai ai zhong xiang y...,"[0.030517578125, -0.023101806640625, -0.002349..."
4,A4-3,PnW,爱，我愿意,十字架上的光芒，温柔又慈祥，\n带着主爱的力量，向着我照亮。\n我的心不再隐藏，完全的摆上，...,shi zi jia shang de guang mang wen rou you ci ...,"[0.031982421875, 0.005176544189453125, -0.0019..."


In [4]:
for d in sorted(os.listdir(root), reverse=True):
    if not d.startswith("2026-"):
        continue
    for f in sorted(os.listdir(f"{root}/{d}")):
        if not bool(re.search(r'[\u4e00-\u9fff]', f)):
            print(f"{root}/{d}/{f}")

/mnt/NextcloudSacmData/sacm.av/files/Recordings/2026-06-14/SQ-ST388.mp3
/mnt/NextcloudSacmData/sacm.av/files/Recordings/2026-05-08 (祷告会)/SQ-ST346.mp3
/mnt/NextcloudSacmData/sacm.av/files/Recordings/2026-05-08 (祷告会)/ZOOM0338.mp3
/mnt/NextcloudSacmData/sacm.av/files/Recordings/2026-05-07/SQ-ST345.mp3
/mnt/NextcloudSacmData/sacm.av/files/Recordings/2026-04-19/SQ-ST323.mp3
/mnt/NextcloudSacmData/sacm.av/files/Recordings/2026-04-19/ZOOM0326.mp3
/mnt/NextcloudSacmData/sacm.av/files/Recordings/2026-03-29/SQ-ST303.mp3
/mnt/NextcloudSacmData/sacm.av/files/Recordings/2026-03-29/ZOOM0300.mp3
/mnt/NextcloudSacmData/sacm.av/files/Recordings/2026-03-07/SQ-ST262.mp3
/mnt/NextcloudSacmData/sacm.av/files/Recordings/2026-03-07/SQ-ST265.mp3
/mnt/NextcloudSacmData/sacm.av/files/Recordings/2026-03-07/ZOOM0258.mp3
/mnt/NextcloudSacmData/sacm.av/files/Recordings/2026-03-01/SQ-ST259.mp3
/mnt/NextcloudSacmData/sacm.av/files/Recordings/2026-03-01/ZOOM0256.mp3
/mnt/NextcloudSacmData/sacm.av/files/Recordings/2026

In [6]:
filepath = f"{root}/2026-03-07/SQ-ST265.mp3"
duration = MP3(filepath).info.length
print(f"{duration=}")
cropped = crop_to_limit(filepath)
audio_file = open(cropped or filepath, "rb")
transcription = client.audio.transcriptions.create(
    # model="gpt-4o-transcribe", 
    model="whisper-1", 
    file=audio_file,
    language="zh",
)
if cropped:
    os.remove(cropped)
lyrics = transcription.text
lyrics

duration=249.96571428571428


'詞曲 李宗盛 詞曲 李宗盛 我的名字 刻劃在你心中 我的臉孔 設定在你眼中 不是因為 我失戀才能 難以適應著 你奇妙寬容和甜蜜 我從頭來 我的名字 刻劃在你心中 我的臉孔 設定在你眼中 不是因為 我失戀才能 難以適應著 你奇妙寬容和甜蜜 雖然有時 我會跌倒軟弱 你卻一直 懦弱不放棄我 用你慈愛 輕輕的擁抱 我在陪你 到最奇妙的海 用你陪伴 所有的甜蜜 彼此在被擁抱 說你的心臟 是無價之寶 一時光 怎會忍住 嚴肅 雖然有時 我會跌倒軟弱 你卻一直 懦弱不放棄我 用你慈愛 輕輕的擁抱 我在陪你 到最奇妙的海 會有你陪伴 所有的甜蜜 彼此在被擁抱 說你的心臟 是無價之寶 一時光 怎會忍住 會有你陪伴 所有的糟糕 一切尊貴榮耀 說你的心臟 是無價之寶 一時光 怎會忍住 嚴肅 字幕由 Amara.org 社群提供'

In [ ]:
titles = {}

query_lyrics = re.sub(r"[，。！、\n]", " ", lyrics)

chunk_size = 120
for start in range(0, len(query_lyrics), chunk_size):
    chunk = query_lyrics[start:start + chunk_size]
    if len(chunk) < 50:
        continue
    # query_embedding = get_embedding(chunk)
    # query_embedding /= np.linalg.norm(query_embedding)
    # scores = embeddings @ query_embedding
    # alpha, beta = (0.1, 0.9) if len(chunk) < chunk_size / 2 else (0.3, 0.7)
    # scores = [weighted_avg(
    #     score, best_window_score(pinyin(chunk), df.pinyin[i], size=len(chunk)//2, step=5), alpha, beta,
    # ) for i, score in enumerate(scores)]
    query_py = pinyin(chunk)
    scores = [best_window_score(query_py, lyric_py, size=len(chunk)//2, step=5) for lyric_py in df.pinyin]
    best_idx = np.argmax(scores)
    best_title = df.iloc[best_idx]["title"]
    best_score = scores[best_idx]
    print(f"[{start}:{start+chunk_size}] {best_title=}, {best_score=}")
    if best_title in titles:
        titles[best_title] = max(titles[best_title], best_score) * 1.2
    else:
        titles[best_title] = best_score

print(f"{titles=}")
final_titles = [title for title, score in titles.items() if score > 0.7]
print(f"Songs: {'_'.join(final_titles)}")

[0:120] best_title='无价至宝', best_score=0.7838983050847458
[120:240] best_title='无价至宝', best_score=0.801762114537445
[240:360] best_title='无价至宝', best_score=0.6696832579185521
titles={'无价至宝': 1.1545374449339207}
Songs: 无价至宝


In [16]:
print(" ".join(df.loc[df.title == "无价至宝"].lyrics.to_list()))

我的名字，刻画在祢心中；我的脸孔，深映在祢眼中。不是因为我努力才能，乃是因着祢奇妙宽容恩典。虽然有时我会跌倒软弱，祢却一直包容不放弃我。用祢慈爱紧紧地拥抱，我赞美祢宝贵奇妙大爱。唯有祢配得所有的赞美，一切尊贵、荣耀。主祢的十架是无价至宝，祢是我赞美的主，耶稣。


In [49]:
chunk = query_lyrics[120:240]
print("Query:", chunk)
query_pinyin = pinyin(chunk)
query_embedding = get_embedding(chunk)
query_embedding /= np.linalg.norm(query_embedding)
scores = embeddings @ query_embedding
for t in ["我的救赎主活着", "当你找到我"]:
    inds = df.loc[df.title == t].index
    for idx in inds:
        print(f"[{idx}] {t}: {df.pinyin[idx]}")
        score = scores[idx]
        fuzz_score = best_window_score(query_pinyin, df.pinyin[idx], size=len(chunk)//2, step=5)
        print(f"{score=}, {fuzz_score=}, {weighted_avg(score, fuzz_score, 0.3, 0.7)}")

Query: 受它逆風生之愛 我知道 我地球輸出永遠火車 我凝固在沉睡 當咆哮響起的那一天 我站前探榮光之邊 當咆哮響起的那一天 我站前探榮光之邊
[319] 我的救赎主活着: wo zhi dao wo de jiu shu zhu huo zhe ta shi yong huo de zhu dang wo zai shen gu mi shi shi ta ling wo zou zheng yi lu wo zhi dao wo de jiu shu zhu huo zhe ta shi yong huo de zhu dang wo zai kuang ye gu du shi ta zuo wo ban wo de deng wo zhi dao wo de jiu shu zhu yong yuan huo zhe wo xin bu zai you lv wo yao zai mei yi ge ri ye zhong song zan ta de feng sheng zhi
score=0.40783736059014475, fuzz_score=0.8494457411744992, 0.3584816134995964
[50] 当你找到我: dang ni zhao dao wo zai sheng ming shi zi lu kou ni kao jin wo ni yong bao wo cong ci ni bu fang shou guo qu de zhong zhong quan ran jiao zai ni shou zhong ni yi zhi wo ni hui fu wo cong ci wo bu fang shou ye zai hei xin zai pi bei wo yang wang ni zhi sheng rong mian wo yi jue ding yong bu hou tui ni da neng zai wo xin jian wo yuan yi sheng jin jin gen sui wo gen sui ni cong gao shan dao di gu wo gen sui
score=0.4132462779654543, fuzz_score=0.7767744268

In [ ]:
def get_titles(filepath):
    print(f"Processing {filepath}")
    duration = MP3(filepath).info.length
    if duration < 60:  # probably noise
        return []

    cropped = crop_to_limit(filepath)
    audio_file = open(cropped or filepath, "rb")
    transcription = client.audio.transcriptions.create(
        model="whisper-1", 
        file=audio_file,
        language="zh",  
    )
    lyrics = transcription.text
    titles = {}
    if cropped:
        os.remove(cropped)

    query_lyrics = re.sub(r"[，。！、\n]", " ", lyrics)

    chunk_size = 120
    for start in range(0, len(query_lyrics), chunk_size):
        chunk = query_lyrics[start:start + chunk_size]
        if len(chunk) < 50:
            continue
        query_embedding = get_embedding(chunk)
        query_embedding /= np.linalg.norm(query_embedding)
        scores = embeddings @ query_embedding
        scores = [weighted_avg(
            score, best_window_score(pinyin(chunk), df.pinyin[i], size=chunk_size, step=chunk_size//4),
            alpha=0.3, beta=0.7,
        ) for i, score in enumerate(scores)]
        best_idx = np.argmax(scores)
        best_title = df.iloc[best_idx]["title"]
        best_score = scores[best_idx]
        # print(f"[{start}:{start+chunk_size}] {best_title=}, {best_score=}")
        if best_title in titles:
            titles[best_title] = max(titles[best_title], best_score) * 1.2
        else:
            titles[best_title] = best_score

    print(f"{titles=}")
    if duration > 5 * 60:
        final_titles = [title for title, score in titles.items() if score > 0.4]
    else:
        best_title = max(titles, key=titles.get)
        final_titles = [best_title] if titles[best_title] > 0.4 else []
    return final_titles

In [ ]:
for d in sorted(os.listdir(root)):
    if not d.startswith("2026-05"):
        continue
    files = sorted(os.listdir(f"{root}/{d}"))
    if any(bool(re.search(r'[\u4e00-\u9fff]', f)) for f in files):  # already renamed
        continue
    zoom_to_sq = match_zoom_to_sq(d)
    zoom_files = [f for f in files if f.startswith("ZOOM")]
    sq_files = [f for f in files if not f.startswith("ZOOM")]
    file_to_titles = {f: get_titles(f"{root}/{d}/{f}") for f in sq_files}
    for zoom_file in zoom_files:
        sq_file = zoom_to_sq.get(zoom_file)
        sq_titles = file_to_titles.get(sq_file, [])
        titles = get_titles(f"{root}/{d}/{zoom_file}") or sq_titles
        if any(t in sq_titles for t in titles):
            titles = sq_titles
        file_to_titles[zoom_file] = titles
    for f, title in file_to_titles.items():
        filename, ext = os.path.splitext(f)
        new_filepath = f"{filename}_{'_'.join(title)}{ext}" if title else f"{filename}{ext}"
        print(f"{f}→{new_filepath}")
        if f != new_filepath:
            os.rename(f"{root}/{d}/{f}", f"{root}/{d}/{new_filepath}")
    !sudo -u www-data php /var/www/html/nextcloud_sacm/occ files:scan --path "sacm.av/files/Recordings/{d}"

Processing /mnt/NextcloudSacmData/sacm.av/files/Recordings/2026-05-07/SQ-ST345.mp3
